# 用 MCP 构建可接入外部资源的 Agent：把工具“插”进模型

## 介绍

Model Context Protocol（MCP）是一套开放协议，用来标准化“应用如何把工具/数据/上下文提供给大语言模型”。你可以把 MCP 理解成 AI 应用的 **USB-C**：

- 以前：每接一个外部系统（数据库、API、文件系统……）都要写一套定制集成
- 现在：外部系统按 MCP 规范“对外提供工具”，你的 Agent 通过统一方式发现并调用

这一节会带你把 MCP 跑通：从“启动一个 MCP Server”到“在自己的 Agent 里调用 MCP 工具”。

## MCP 架构概览

![MCP Architecture](../images/mcp_architecture.png)

MCP 采用典型的 client-server 架构：

- **Host**：你的 AI 应用（例如 Claude Desktop、Cursor、或你自己写的 agent）
- **Client**：负责与 MCP server 建立/维护连接
- **Server**：轻量程序，对外暴露能力（tools / prompts / resources）
- **Data Sources**：server 可以连接的真实数据源（本地文件、数据库、远程 API 等）

通信通常基于 JSON-RPC 2.0（不同传输方式可能是 stdio / HTTP 等）。

## 先体验一下 MCP

如果你想先“用一下现成的 MCP server”来获得直觉，可以先看 MCP 的用户向快速上手文档：

- `https://modelcontextprotocol.io/quickstart/user`

你可以先体验“工具出现在聊天框里，模型可以直接调用”，再回头看本节的实现细节。

## 构建你的 MCP Server（以加密货币价格查询为例）

这一部分会做一个“加密货币价格查询”的 MCP server（使用 CoinGecko API），然后接入 Claude Desktop。

> 说明：这一段需要外网 API 与本机安装环境（`uv`、Claude Desktop）。如果你只想在仓库内快速跑通 MCP 工具调用，直接看后面基于本地 `math_server.py` 的演示即可。

### 环境准备（终端执行）

使用 `uv` 作为包管理器（终端里执行，不要放到 notebook cell 里）：

```bash
curl -LsSf https://astral.sh/uv/install.sh | sh

mkdir mcp-crypto-server
cd mcp-crypto-server
uv init

uv venv
source .venv/bin/activate

uv add "mcp[cli]" httpx
```

### 启动 MCP Server

在 `all_agents_tutorials/scripts/mcp_server.py` 提供了一个示例 server（加密货币价格查询）。你可以把它复制到你的 `mcp-crypto-server` 目录后运行：

```bash
cp ../scripts/mcp_server.py .
uv run mcp_server.py
```

### 接入 Claude Desktop（可选）

Claude Desktop 允许你在配置文件里声明 MCP server，然后在聊天里看到工具入口。

示例配置（把路径改成你自己的绝对路径）：

```json
{
  "mcpServers": {
    "crypto-price-tracker": {
      "command": "/ABSOLUTE/PATH/TO/uv",
      "args": [
        "--directory",
        "/ABSOLUTE/PATH/TO/GenAI_Agents/all_agents_tutorials/mcp-crypto-server",
        "run",
        "mcp_server.py"
      ]
    }
  }
}
```

你会在界面里看到工具入口（示意图）：

![Claude Desktop connected with MCP](../images/Claude_Desktop_with_MCP.png)

示例效果：

![Track Bitcoin price with MCP](../images/track_bitcoin_price_with_mcp.png)

![Track Crypto Market Data with MCP](../images/track_crypto_market_data_with_mcp.png)

## 通过 MCP 执行工具的自定义 Agent

接下来在 notebook 里实现“Host + 调用链路”，并把 MCP tools 接到 LangGraph。

### 理解整体架构

这一部分会搭一个简单的 Agent，它可以：
1. 充当 MCP Host
2. 从 MCP Server 发现可用工具
3. 根据用户问题判断该用哪个工具
4. 用合适的参数执行工具
5. 把工具结果整理成对用户有帮助的回答

这个结构在很多系统里都很常见：
- **Discovery Phase**：发现有哪些工具
- **Planning Phase**：决定要调用哪个工具
- **Execution Phase**：执行工具
- **Interpretation Phase**：解释结果

下面是一张简单的工作流示意图：

![Customized MCP Host](../images/customized_mcp_host.png)

运行下面代码前的提醒：
- 如果你连接的是一个“独立运行”的 MCP server，请先把 server 启动起来；否则将无法发现/执行工具。
- 本节的 stdio 示例会由客户端启动 `scripts/math_server.py`（不需要你手动开一个常驻服务）。

- MCP 连接与工具封装：`langchain-mcp-adapters`（`MultiServerMCPClient`）
- 模型：`ChatOpenAI`
- 执行框架：LangGraph 的 `bind_tools + ToolNode + 条件边`

为了让示例开箱即跑，这里用一个 **本地 stdio MCP server**（`scripts/math_server.py`）来演示。

### 准备：导入依赖并加载环境变量

你只需要保证环境里已安装：`mcp`、`langchain-mcp-adapters`、`langgraph` 等。

In [1]:
import os
from pathlib import Path
from typing import Annotated, TypedDict

from dotenv import load_dotenv
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode


load_dotenv("../.env")

# 约定：运行本 notebook 时，当前工作目录为 all_agents_tutorials/
MATH_SERVER = Path("scripts/math_server.py").resolve()
MATH_SERVER

PosixPath('/data/tp/006-project/002-paper_qa/references/GenAI_Agents_CN/all_agents_tutorials/scripts/math_server.py')

### 工具发现：构建 MCP Host

这里做的事很简单：

1) 以 stdio 的方式启动 `math_server.py`
2) `get_tools()` 拉取 server 暴露的工具列表
3) 得到的 `tools` 是 LangChain 的 `BaseTool`，可以直接塞进 LangGraph 的 tool calling 流程

In [2]:
client = MultiServerMCPClient(
    {
        "math": {
            "transport": "stdio",
            "command": "python",
            "args": [str(MATH_SERVER)],
        }
    }
)

tools = await client.get_tools()
[t.name for t in tools]

['add', 'mul']

### 工具执行：实现 MCP Client

下面演示不经过模型，直接调用 MCP 工具（相当于先验证 client 能执行 tool）。


In [3]:
tools_by_name = {t.name: t for t in tools}

async def execute_tool(tool_name: str, arguments: dict):
    return await tools_by_name[tool_name].ainvoke(arguments)

list(tools_by_name.keys())

['add', 'mul']

In [4]:
add_result = await execute_tool("add", {"a": 3, "b": 5})
mul_result = await execute_tool("mul", {"a": 8, "b": 12})
add_result, mul_result

([{'type': 'text',
   'text': '8',
   'id': 'lc_4535325c-4f86-4f1f-a1f2-c7deca74e60e'}],
 [{'type': 'text',
   'text': '96',
   'id': 'lc_e3dd39f5-d210-4a05-baaf-90a381fc805c'}])

### 集成 AI：让模型通过 MCP 调用工具（call_model → ToolNode → call_model）

我们用你已经熟悉的模式：

- `llm.bind_tools(tools)`：让模型“看见并会调用这些工具”
- `ToolNode(tools)`：负责执行 tool_calls 并返回 ToolMessage
- 条件边：如果 `AIMessage.tool_calls` 非空就走 tools，否则结束


In [5]:
llm = ChatOpenAI(
    model="deepseek-v4-flash-0731",
    api_key=os.environ.get("DASHSCOPE_API_KEY"),
    base_url=os.environ.get("DASHSCOPE_BASE_URL"),
    temperature=0,
)

llm_with_tools = llm.bind_tools(tools)


class State(TypedDict):
    messages: Annotated[list, add_messages]


def call_model(state: State) -> dict:
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}


tool_node = ToolNode(tools)


def should_continue(state: State):
    last = state["messages"][-1]
    tool_calls = getattr(last, "tool_calls", None) or []
    return "tools" if tool_calls else END


builder = StateGraph(State)
builder.add_node("call_model", call_model)
builder.add_node("tools", tool_node)
builder.add_edge(START, "call_model")
builder.add_conditional_edges("call_model", should_continue, {"tools": "tools", END: END})
builder.add_edge("tools", "call_model")

app = builder.compile()

### 测试：让模型用 MCP 工具算一道题

你会在 messages trace 里看到：

- `AIMessage` 里出现对 MCP 工具的调用（tool_calls）
- `ToolMessage` 返回工具执行结果
- 最后 `AIMessage` 用结果给出答案

In [6]:
result = await app.ainvoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Use tools to compute (3 + 5) * 12. Return only the number.",
            }
        ]
    }
)

for i, m in enumerate(result["messages"]):
    t = type(m).__name__
    name = getattr(m, "name", "")
    content = getattr(m, "content", "")
    if isinstance(content, str) and len(content) > 200:
        content = content[:200] + " ..."
    if t == "ToolMessage":
        print(f"[{i:02d}] {t}({name}): {content!r}")
    else:
        print(f"[{i:02d}] {t}: {content!r}")

[00] HumanMessage: 'Use tools to compute (3 + 5) * 12. Return only the number.'
[01] AIMessage: ''
[02] ToolMessage(add): [{'type': 'text', 'text': '8', 'id': 'lc_0334e929-79d8-4df2-aee6-0e234ac531e9'}]
[03] AIMessage: ''
[04] ToolMessage(mul): [{'type': 'text', 'text': '96', 'id': 'lc_027679d2-7b35-4fcd-b8a8-f8b368c557cc'}]
[05] AIMessage: '96'


### 构建一个交互式的 MCP Host 界面

下面提供一个最小交互循环：你输入问题，模型会在需要时调用 MCP 工具。

- 输入 `exit` / `quit` 退出。


In [7]:
async def chat_session():
    while True:
        q = input("Question (or 'exit'): ").strip()
        if q.lower() in {"exit", "quit"}:
            break
        result = await app.ainvoke({"messages": [{"role": "user", "content": q}]})
        print(result["messages"][-1].content)


await chat_session()

5+6的计算结果是 **11**。


## 检查理解（含答案）

1) **MCP 解决的核心问题是什么？**
- 答：把各种外部系统（数据源/工具/API）用统一协议暴露给模型，让“接入新能力”从定制开发变成标准连接。

2) **`get_tools()` 拿到的是什么？为什么能直接给 LangGraph 用？**
- 答：`langchain-mcp-adapters` 会把 MCP tools 封装成 LangChain 的 `BaseTool`，因此可以直接参与 tool calling 流程（`bind_tools` / `ToolNode`）。

3) **为什么还需要 `ToolNode`？**
- 答：`bind_tools` 只负责让模型“产出 tool_calls”；真正执行 tool_calls 并把结果写成 `ToolMessage` 回填到 messages，需要 `ToolNode` 来做。


## 总结

这一节你学到的是：

- MCP 把“外部能力”封装成标准工具接口
- 你的 Agent 通过 MCP client 动态发现 tools
- 拿到的 tools 可以像普通工具一样接进 LangGraph（`bind_tools` + `ToolNode`）
